# 📖 Notebook 2: Secondary Indexes (GSI & LSI)

What happens when you need to query your data by an attribute that isn't the partition key? You create a **secondary index**.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why secondary indexes exist and when you need them
- The difference between Global Secondary Indexes (GSI) and Local Secondary Indexes (LSI)
- How to create and query each type of index
- The trade-offs (cost, consistency, storage) of each index type
- How indexes work under the hood

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 03-technologies/databases/dynamodb
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import boto3
from boto3.dynamodb.conditions import Key
from botocore.exceptions import ClientError
import json
import time

dynamodb = boto3.resource(
    "dynamodb",
    endpoint_url="http://localhost:8000",
    region_name="us-east-1",
    aws_access_key_id="local",
    aws_secret_access_key="local",
)

client = boto3.client(
    "dynamodb",
    endpoint_url="http://localhost:8000",
    region_name="us-east-1",
    aws_access_key_id="local",
    aws_secret_access_key="local",
)

try:
    client.list_tables()
    print("✅ Connected to DynamoDB Local")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("   Run: docker-compose up -d")

## 🤔 The Problem: Querying by Non-Key Attributes

Imagine a chat application. Our main table looks like this:

| chat_id (PK) | message_id (SK) | sender | text | timestamp |
|---|---|---|---|---|
| general | msg-001 | Alice | Hey everyone! | 2024-01-15T10:00:00Z |
| general | msg-002 | Bob | Hi Alice! | 2024-01-15T10:01:00Z |
| engineering | msg-001 | Dave | Deploy done | 2024-01-15T11:00:00Z |

We can easily query: *"Get all messages in the general chat"* (uses partition key).

But what about: *"Get all messages sent by Alice across ALL chats"*?

The `sender` attribute is not part of the primary key, so we'd have to **Scan** the entire table and filter. That's slow and expensive! This is where secondary indexes come in.

## 🌍 Part 1: Global Secondary Index (GSI)

A **Global Secondary Index** lets you query data using a completely different partition key.

Think of it like creating an alternative phone book:
- The main phone book is organized by **last name** (your table's partition key)
- A GSI is like a second phone book organized by **city** (a different attribute)
- Both contain the same people, just organized differently

### Key facts about GSIs:
- Uses a **different partition key** (and optional sort key) from the base table
- Data is stored on **separate physical partitions** (it's like a separate table)
- Updates from the base table are replicated **asynchronously** (eventually consistent only)
- Can be **added or removed** at any time
- Up to **20 GSIs** per table

In [ ]:
# Create a ChatMessages table WITH a GSI on 'sender'

TABLE_NAME = "ChatMessagesWithGSI"

try:
    dynamodb.Table(TABLE_NAME).delete()
    dynamodb.Table(TABLE_NAME).wait_until_not_exists()
except ClientError:
    pass

table = dynamodb.create_table(
    TableName=TABLE_NAME,
    KeySchema=[
        {"AttributeName": "chat_id", "KeyType": "HASH"},
        {"AttributeName": "message_id", "KeyType": "RANGE"},
    ],
    AttributeDefinitions=[
        {"AttributeName": "chat_id", "AttributeType": "S"},
        {"AttributeName": "message_id", "AttributeType": "S"},
        {"AttributeName": "sender", "AttributeType": "S"},
    ],
    GlobalSecondaryIndexes=[
        {
            "IndexName": "SenderIndex",
            "KeySchema": [
                {"AttributeName": "sender", "KeyType": "HASH"},      # GSI partition key
                {"AttributeName": "message_id", "KeyType": "RANGE"},  # GSI sort key
            ],
            "Projection": {"ProjectionType": "ALL"},  # Include ALL attributes in the index
        },
    ],
    BillingMode="PAY_PER_REQUEST",
)

table.wait_until_exists()
print(f"✅ Created table '{TABLE_NAME}'")
print(f"   Base table key: chat_id (PK) + message_id (SK)")
print(f"   GSI 'SenderIndex': sender (PK) + message_id (SK)")
print()
print("💡 The GSI lets us query by 'sender' — something impossible with the base table key.")

In [ ]:
# Insert test data

messages = [
    {"chat_id": "general",     "message_id": "msg-001", "sender": "Alice", "text": "Hey everyone!",    "timestamp": "2024-01-15T10:00:00Z"},
    {"chat_id": "general",     "message_id": "msg-002", "sender": "Bob",   "text": "Hi Alice!",        "timestamp": "2024-01-15T10:01:00Z"},
    {"chat_id": "general",     "message_id": "msg-003", "sender": "Alice", "text": "How is everyone?", "timestamp": "2024-01-15T10:05:00Z"},
    {"chat_id": "engineering", "message_id": "msg-001", "sender": "Dave",  "text": "Deploy is done",   "timestamp": "2024-01-15T11:00:00Z"},
    {"chat_id": "engineering", "message_id": "msg-002", "sender": "Alice", "text": "Great, thanks!",   "timestamp": "2024-01-15T11:01:00Z"},
    {"chat_id": "engineering", "message_id": "msg-003", "sender": "Bob",   "text": "Nice work!",       "timestamp": "2024-01-15T11:02:00Z"},
    {"chat_id": "random",      "message_id": "msg-001", "sender": "Alice", "text": "Fun fact!",        "timestamp": "2024-01-15T12:00:00Z"},
]

with table.batch_writer() as batch:
    for msg in messages:
        batch.put_item(Item=msg)

print(f"✅ Inserted {len(messages)} messages")
print(f"   Alice sent messages in: general, engineering, random")

In [ ]:
# Query the GSI: "Get ALL messages sent by Alice across ALL chats"

response = table.query(
    IndexName="SenderIndex",  # Tell DynamoDB to use the GSI
    KeyConditionExpression=Key("sender").eq("Alice"),
)

print("📨 GSI Query: All messages by Alice (across all chats)")
print("=" * 60)
for item in response["Items"]:
    print(f"  [{item['chat_id']}] [{item['message_id']}] {item['text']}")

print(f"\n📊 Found {response['Count']} messages by Alice")
print()
print("💡 Without the GSI, we'd have to Scan the entire table and filter.")
print("   With the GSI, this is a targeted Query — fast and cheap!")

In [ ]:
# Compare: GSI Query vs Scan + Filter

# Method 1: GSI Query (efficient)
start = time.time()
for _ in range(100):
    table.query(
        IndexName="SenderIndex",
        KeyConditionExpression=Key("sender").eq("Alice"),
    )
gsi_time = (time.time() - start) * 1000 / 100

# Method 2: Scan + Filter (expensive)
from boto3.dynamodb.conditions import Attr

start = time.time()
for _ in range(100):
    table.scan(
        FilterExpression=Attr("sender").eq("Alice"),
    )
scan_time = (time.time() - start) * 1000 / 100

print("⏱️  Performance Comparison (100 iterations, avg per call)")
print("=" * 60)
print(f"   GSI Query:     {gsi_time:.2f} ms  (reads only Alice's partition)")
print(f"   Scan + Filter: {scan_time:.2f} ms  (reads EVERY item, then filters)")
print()
print("💡 With a small dataset the difference is small, but at scale:")
print("   - 1M items: GSI reads ~100 items. Scan reads all 1M.")
print("   - Cost: Scan consumes RCU for ALL items, GSI only for matching items.")

## 📍 Part 2: Local Secondary Index (LSI)

A **Local Secondary Index** uses the **same partition key** as the base table but a **different sort key**.

Think of it like reorganizing books on the SAME shelf:
- Base table: books on each shelf sorted by **title**
- LSI: same shelves, but now also sorted by **page count**

### Key facts about LSIs:
- **Same partition key** as the base table, different sort key
- Data is stored on the **same partition** (co-located with the base table)
- Updates are **synchronous** — always in sync with the base table
- Supports **strongly consistent reads** (unlike GSI!)
- Must be created **at table creation time** (cannot be added later!)
- Up to **5 LSIs** per table
- 10 GB size limit per partition

In [ ]:
# Create a table with an LSI
# Scenario: Chat messages sorted by timestamp (instead of message_id)

LSI_TABLE = "ChatMessagesWithLSI"

try:
    dynamodb.Table(LSI_TABLE).delete()
    dynamodb.Table(LSI_TABLE).wait_until_not_exists()
except ClientError:
    pass

lsi_table = dynamodb.create_table(
    TableName=LSI_TABLE,
    KeySchema=[
        {"AttributeName": "chat_id", "KeyType": "HASH"},
        {"AttributeName": "message_id", "KeyType": "RANGE"},
    ],
    AttributeDefinitions=[
        {"AttributeName": "chat_id", "AttributeType": "S"},
        {"AttributeName": "message_id", "AttributeType": "S"},
        {"AttributeName": "timestamp", "AttributeType": "S"},
    ],
    LocalSecondaryIndexes=[
        {
            "IndexName": "TimestampIndex",
            "KeySchema": [
                {"AttributeName": "chat_id", "KeyType": "HASH"},      # Same PK as base table
                {"AttributeName": "timestamp", "KeyType": "RANGE"},   # Different sort key!
            ],
            "Projection": {"ProjectionType": "ALL"},
        },
    ],
    BillingMode="PAY_PER_REQUEST",
)

lsi_table.wait_until_exists()
print(f"✅ Created table '{LSI_TABLE}'")
print(f"   Base table key: chat_id (PK) + message_id (SK)")
print(f"   LSI 'TimestampIndex': chat_id (PK) + timestamp (SK)")
print()
print("⚠️  LSIs MUST be defined at table creation time — you can't add them later!")

In [ ]:
# Insert messages with out-of-order timestamps
# (message_id order doesn't match timestamp order)

messages = [
    {"chat_id": "general", "message_id": "msg-001", "sender": "Alice", "text": "Morning!",       "timestamp": "2024-01-15T10:00:00Z", "num_attachments": 0},
    {"chat_id": "general", "message_id": "msg-002", "sender": "Bob",   "text": "Check this out",  "timestamp": "2024-01-15T09:55:00Z", "num_attachments": 3},
    {"chat_id": "general", "message_id": "msg-003", "sender": "Carol", "text": "Hello!",          "timestamp": "2024-01-15T10:10:00Z", "num_attachments": 1},
    {"chat_id": "general", "message_id": "msg-004", "sender": "Alice", "text": "See photo",       "timestamp": "2024-01-15T09:50:00Z", "num_attachments": 5},
    {"chat_id": "general", "message_id": "msg-005", "sender": "Bob",   "text": "On my way",       "timestamp": "2024-01-15T10:15:00Z", "num_attachments": 0},
]

with lsi_table.batch_writer() as batch:
    for msg in messages:
        batch.put_item(Item=msg)

print(f"✅ Inserted {len(messages)} messages")
print("   Note: message_id order ≠ timestamp order (msg-004 is earliest!)")

In [ ]:
# Query using the base table (sorted by message_id)

response = lsi_table.query(
    KeyConditionExpression=Key("chat_id").eq("general"),
)

print("📨 Base Table Query (sorted by message_id):")
print("=" * 60)
for item in response["Items"]:
    print(f"  [{item['message_id']}] {item['timestamp']} — {item['sender']}: {item['text']}")

In [ ]:
# Query using the LSI (sorted by timestamp)

response = lsi_table.query(
    IndexName="TimestampIndex",
    KeyConditionExpression=Key("chat_id").eq("general"),
)

print("📨 LSI Query (sorted by timestamp):")
print("=" * 60)
for item in response["Items"]:
    print(f"  {item['timestamp']} [{item['message_id']}] — {item['sender']}: {item['text']}")

print()
print("💡 Same partition (general), but items come back in timestamp order!")
print("   The LSI maintains a separate B-tree sorted by 'timestamp' within each partition.")

In [ ]:
# LSI range query: Get messages in a time window

response = lsi_table.query(
    IndexName="TimestampIndex",
    KeyConditionExpression=(
        Key("chat_id").eq("general") &
        Key("timestamp").between("2024-01-15T10:00:00Z", "2024-01-15T10:15:00Z")
    ),
)

print("📨 LSI Query: Messages between 10:00 and 10:15")
print("=" * 60)
for item in response["Items"]:
    print(f"  {item['timestamp']} — {item['sender']}: {item['text']}")

print(f"\n📊 Found {response['Count']} messages in that time window")

## 📊 Part 3: GSI vs LSI Comparison

| Feature | GSI | LSI |
|---------|-----|-----|
| **Partition Key** | Different from base table | Same as base table |
| **Sort Key** | Optional | Required (different from base table) |
| **Storage** | Separate partitions | Co-located with base table |
| **Consistency** | Eventually consistent only | Supports strongly consistent reads |
| **Update Sync** | Asynchronous | Synchronous |
| **Creation** | Anytime | Table creation time only |
| **Max Count** | 20 per table | 5 per table |
| **Size Limit** | No limit | 10 GB per partition |
| **Throughput** | Own capacity | Shares with base table |

### When to use which?

- **Use GSI** when you need to query by a **completely different attribute** across all partitions  
  Example: "Find all messages by Alice" → GSI on `sender`

- **Use LSI** when you need an **alternative sort order** within the same partition  
  Example: "Sort messages by timestamp instead of message_id" → LSI on `timestamp`

## 🔬 Part 4: Projection Types (What Gets Copied to the Index)

When you create an index, you choose which attributes are **projected** (copied) into it:

| Projection Type | What's Included | Storage Cost | Use When |
|----------------|----------------|-------------|----------|
| `ALL` | All attributes | Highest | Need all data from index queries |
| `KEYS_ONLY` | Only key attributes | Lowest | Just need to know which items match |
| `INCLUDE` | Keys + specified attributes | Medium | Need some but not all attributes |

In [ ]:
# Create a table with different projection types to see the difference

PROJ_TABLE = "ProjectionDemo"

try:
    dynamodb.Table(PROJ_TABLE).delete()
    dynamodb.Table(PROJ_TABLE).wait_until_not_exists()
except ClientError:
    pass

proj_table = dynamodb.create_table(
    TableName=PROJ_TABLE,
    KeySchema=[
        {"AttributeName": "pk", "KeyType": "HASH"},
        {"AttributeName": "sk", "KeyType": "RANGE"},
    ],
    AttributeDefinitions=[
        {"AttributeName": "pk", "AttributeType": "S"},
        {"AttributeName": "sk", "AttributeType": "S"},
        {"AttributeName": "gsi_pk", "AttributeType": "S"},
    ],
    GlobalSecondaryIndexes=[
        {
            "IndexName": "AllProjection",
            "KeySchema": [{"AttributeName": "gsi_pk", "KeyType": "HASH"}],
            "Projection": {"ProjectionType": "ALL"},
        },
        {
            "IndexName": "KeysOnlyProjection",
            "KeySchema": [{"AttributeName": "gsi_pk", "KeyType": "HASH"}],
            "Projection": {"ProjectionType": "KEYS_ONLY"},
        },
        {
            "IndexName": "IncludeProjection",
            "KeySchema": [{"AttributeName": "gsi_pk", "KeyType": "HASH"}],
            "Projection": {
                "ProjectionType": "INCLUDE",
                "NonKeyAttributes": ["name"],  # Only include 'name'
            },
        },
    ],
    BillingMode="PAY_PER_REQUEST",
)

proj_table.wait_until_exists()

# Insert a sample item
proj_table.put_item(Item={
    "pk": "item-1",
    "sk": "detail",
    "gsi_pk": "category-A",
    "name": "Widget",
    "price": 29.99,
    "description": "A very long description that takes up storage space...",
})

# Query each index and see what attributes come back
for index_name in ["AllProjection", "KeysOnlyProjection", "IncludeProjection"]:
    response = proj_table.query(
        IndexName=index_name,
        KeyConditionExpression=Key("gsi_pk").eq("category-A"),
    )
    attrs = list(response["Items"][0].keys()) if response["Items"] else []
    print(f"\n📦 {index_name}:")
    print(f"   Attributes returned: {attrs}")

print()
print("💡 KEYS_ONLY saves storage but returns minimal data.")
print("   If you need an attribute not in the projection, DynamoDB must")
print("   fetch the full item from the base table (extra read cost).")

## 🎯 Key Takeaways

1. **GSI** = query by a different partition key (cross-partition search)
2. **LSI** = query with a different sort order within the same partition
3. **GSIs are eventually consistent** — there's a small delay between base table writes and index updates
4. **LSIs support strong consistency** — reads can reflect the latest write immediately
5. Choose **projection type** carefully — ALL is convenient but costs more storage
6. **Plan LSIs early** — they must be defined at table creation time

### Under the Hood
- **GSI**: A separate internal table with its own partition scheme. DynamoDB asynchronously replicates changes from the base table.
- **LSI**: A separate B-tree within each base table partition. DynamoDB synchronously updates it with every write.

### Next Up
In the next notebook, we'll learn about **Single-Table Design** — how to model multiple entity types in one DynamoDB table for efficient access patterns.